In [14]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple
import numpy as np
from statsmodels.tsa.ar_model import AutoReg
from sklearn.linear_model import LinearRegression
import cvxpy as cp

In [3]:
@dataclass
class Universe:
  Bonds:List[str]
  Futures:List[str]
  Commodities:List[str]
  High_Beta:List[str]
  High_Yield:List[str]
  Sat_Defensive:List[str]

In [25]:
class DataStore:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def _get_data(
      self,
      universe:dict,
      start:str,
      end:str,
      interval:str="1d",
      benchmark:str="^GSPC"
  ):
    tickers = list(universe.values())
    self.benchmark_ticker = benchmark
    df_path = f"portfolio_{start}_{end}.parquet"

    if not os.path.exists(df_path):
      tickers.append(benchmark)
      df = yf.download(tickers, start, end, interval, group_by="tickers")

      df.to_parquet(df_path)

    else:
      df = pd.read_parquet(df_path)

    bench_data = df["^GSPC"]["Close"]
    data_raw = df.drop(columns=["^GSPC"])

    benchmark = bench_data.pct_change().dropna()
    self.universe = data_raw.columns

    return data_raw, benchmark

  def plot_data(self):
    (np.cumsum(self.returns_raw * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

  def plot_benchmark(self):
    (np.cumsum(self.benchmark * 100, axis=0) + 100).plot(figsize=(15, 10))
    plt.show()

In [26]:
class Filter:
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def corr_filter(self, returns):
    corr_matrix = returns.corr()
    std = returns.std()

    sharpe = (np.mean(returns)-self.rf) / std

    drop = []
    for ticker in returns.columns:
      if ticker in drop:
        continue
      ticker_idx = returns.columns.get_loc(ticker)

      for i in returns.columns:
        if ticker == i or i in drop:
          continue

        i_idx = returns.columns.get_loc(i)
        if corr_matrix.iloc[ticker_idx, i_idx] > self.corr_threshold:
          if self.debug:
            print(corr_matrix.iloc[ticker_idx, i_idx])
          if sharpe[ticker] > sharpe[i]:
            drop.append(i)
          else:
            drop.append(ticker)

    tmp = returns.drop(columns=drop)

    return returns.columns

  def abs_return_unq_filter(
    self,
    returns:pd.DataFrame,
    benchmark:pd.Series,
    rfr:float=0.003,
    max_r2:float=0.15,
    min_sharpe:float=0.30,
    min_iur:float=0.70,
    T:int=252
  )->list:
    model, X, y = self._run_regression(
        [returns, benchmark]
    )
    beta = model.coef_[0]
    r2 = model.score(X, y)

    pred = model.predict(X)
    residuals = y - pred

    iur = np.var(residuals) / np.var(y) if np.var(y) > 0 else 0

    ann_res_mean = np.mean(residuals) * 252
    ann_res_std = np.std(residuals) * np.sqrt(252)
    residual_sharpe = ann_res_mean / ann_res_std if ann_res_std > 0 else -np.inf

    passed_r2 = r2 <= max_r2
    passed_iur = iur >= min_iur
    passed_sharpe = residual_sharpe >= min_sharpe

    is_qualified = passed_r2 and passed_iur and passed_sharpe

    return is_qualified

  def _run_regression(self, asset_list):
    df = pd.concat(
      asset_list,
      axis=1
    ).dropna()
    df.columns = ['asset', 'benchmark']

    X = df[['benchmark']].values
    y = df['asset'].values

    model = LinearRegression().fit(X, y)

    return model, X, y

  def commodity_factor_filter(
      asset_returns: pd.Series,
      market_returns: pd.Series,
      cpi_surprises: pd.Series,
      max_equity_corr: float = 0.35,
      min_cpi_corr: float = 0.20,
      max_beta_std: float = 0.25
  ) -> list:
    df = pd.concat([asset_returns, market_returns, cpi_surprises], axis=1).dropna()
    df.columns = ['Asset', 'Market', 'CPI_Surprise']

    equity_corr = df['Asset'].corr(df['Market'])
    pass_equity = abs(equity_corr) <= max_equity_corr

    cpi_corr = df['Asset'].corr(df['CPI_Surprise'])
    pass_cpi = cpi_corr >= min_cpi_corr

    rolling_cov = df['Asset'].rolling(60).cov(df['Market'])
    rolling_var = df['Market'].rolling(60).var()
    rolling_beta = (rolling_cov / rolling_var).dropna()
    beta_std = rolling_beta.std()
    pass_beta = beta_std <= max_beta_std

    is_qualified = pass_equity and pass_cpi and pass_beta

    return is_qualified


  def high_beta_filter(
    self,
    returns:pd.DataFrame,
    benchmark:pd.DataFrame,
    min_sector_beta:float=1.2
  ):
    model, X, y = self._run_regression(
        [returns, benchmark]
    )
    beta = model.coef_[0]
    passed_beta = beta > min_sector_beta

    return passed_beta

  def downside_capture_filter(self, returns, benchmark):
    down_mask = returns < 0

    if not down_mask.any():
        return 1.0

    asset_down_compound = np.prod(1 + returns[down_mask]) - 1
    bench_down_compound = np.prod(1 + benchmark[down_mask]) - 1

    return asset_down_compound / bench_down_compound


In [ ]:
class Portfolio(DataStore):
  def __init__(self, debug:bool=False, **kwargs):
    super().__init__(
        debug=debug,
        **kwargs
    )
    self.debug = debug

  def get_data(self, tickers, start, end, benchmark="^GSPC"):
    data, benchmark = self._get_data(
        tickers,
        benchmark=benchmark
    )

    cpi = self.get_cpi(start, end)

    return data, benchmark, cpi

  def filter_universe(
    self,
    universe:dict,
    data:pd.DataFrame,
    benchmark:pd.DataFrame,
    cpi:pd.DataFrame,
    filter_params:dict
  ):
    returns = {}
    for ticker in data.columns:
      returns[ticker] = data[ticker]["Close"].pct_change().dropna()

    returns = pd.DataFrame(returns)
    delta_cpi = cpi - cpi.shift(-1)





